In [1]:
import pandas as pd
import numpy as np
from enum import Enum
import cv2
from collections import deque
import json

def obtain_estimates(video_ID):
    vid = cv2.VideoCapture(f'/home/edisonz/maneuver_heuristics25/clips/Parking-clip{video_ID}.mp4')
    frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))  # Max x-coordinate
    frame_height = 2*int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))//3  # Max y-coordinate
    # ----------------------------------------------------------------------------------------------------------------------------

    # Enter hyperparameters (adjustable)
    start_entry_ratio = 0.5 # 0 = zone based | 1 = peak based 
    end_entry_ratio = 1.0 # 0 = front parking zone | 1 = rear parking zone

    # Exit hyperparameters (adjustable)
    start_exit_ratio = 0 # 0 = rear parking zone | 1 = front parking zone
    end_exit_ratio = 1.0 # 0 = peak based | 1 = zone based

    # low motion hyperparameters 
    movement_threshold = 1.0

    # Peak hyperparameters 
    tA = 5
    aA = np.pi/6

    # Load data
    vehicle_df = pd.read_csv(f'clips/Parking-clip{video_ID}.csv')
    maneuvers = pd.read_csv(f'clip-annotations/maneuver{video_ID}.csv')
    start_frame, end_frame = np.median(maneuvers["start_frame"]), np.median(maneuvers["end_frame"])
    driving_df = vehicle_df[vehicle_df["in_parking_zone"]==0]
    driving_coords = driving_df[['x', 'y']].values
    parking_df = vehicle_df[vehicle_df["in_parking_zone"]==1]
    parking_coords = parking_df[['x', 'y']].values
    start_frame = int(round(start_frame))
    end_frame = int(round(end_frame))
    
    region_code = None
    with open("mappings.json", 'r') as f:
        data = json.load(f)
        region_code = data.get(f'{video_ID}')
    parking_lines = f'lines{region_code}.json'

    class ManeuverType(Enum):
        ENT = 0
        EXT = 1

    maneuver_type = ManeuverType.ENT if video_ID[-3:].lower()=='ent' else ManeuverType.EXT

    # --------------------------------------------------------------------------------------------------------------------------

    # Determine if direction reversal in x or y occured
    def is_reversal(r1, r2):
        angle_diff = np.abs(np.arctan2(r1[1], r1[0]) - np.arctan2(r2[1], r2[0]))
        if angle_diff < aA:
            return False  # No significant angle change
        return (r1[0] * r2[0] < 0) or (r1[1] * r2[1] < 0)  # Opposite signs in x or y

    # Determine if vehicle is hugging the border of the frame
    def on_border(df_row):
        height = df_row["height"]
        width = df_row["width"]
        x = df_row["x"]
        y = df_row["y"]
        left_x = int(x-width//2)
        right_x = int(x+width//2)
        top_y = int(y-height//2)
        bottom_y = int(y+height//2)
        return (left_x==0 or top_y==0 or right_x==frame_width or bottom_y==frame_height)

    def load_lines(json_file):
        with open(json_file, 'r') as f:
            data = json.load(f)
        lines = [list(map(tuple, line)) for line in data['lines']]
        return lines

    def compute_vanishing_point(lines):
        A = []
        b = []
        for (x1, y1), (x2, y2) in lines:
            if x1 == x2:  # vertical line
                m = np.inf
                c = x1
            else:
                m = (y2 - y1) / (x2 - x1)
                c = y1 - m * x1

            if m == 0:
                continue

            if np.isinf(m):
                A.append([1, 0])
                b.append(c)
            else:
                A.append([-m, 1])
                b.append(c)

        if len(A) < 1:
            return None
        A = np.array(A)
        b = np.array(b)
        vp, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        return vp

    def fit_line_through_furthest_endpoints(lines, vp):
        if vp is None:
            return None, None, None
        vp = np.array(vp)
        furthest_points = []

        for (x1, y1), (x2, y2) in lines:
            p1 = np.array([x1, y1])
            p2 = np.array([x2, y2])
            d1 = np.linalg.norm(p1 - vp)
            d2 = np.linalg.norm(p2 - vp)
            if d1 > d2:
                furthest_points.append(p1)
            else:
                furthest_points.append(p2)

        furthest_points = np.array(furthest_points)
        if len(furthest_points) < 2:
            return None, None, None
        m, c = np.polyfit(furthest_points[:, 0], furthest_points[:, 1], 1)
        return m, c, furthest_points

    def find_shortest_distance(point, json_file):
        """
        Compute the shortest distance for a given point:
        - If the point lies between two parking lines, return the sum of perpendicular distances to the nearest lines on either side.
        - If the point does not lie between two lines, return the perpendicular distance to the closest line (min perp dist among all lines).
        Lines include both original from json and extrapolated ones.

        Args:
            point (tuple): (x, y) coordinates of the point
            json_file (str): Path to the JSON file containing line data

        Returns:
            float: Shortest distance (sum of perpendicular distances or min distance to closest line), or None if no lines exist
        """
        lines = load_lines(json_file)
        x_point, y_point = point
        if not lines:
            return None

        vp = compute_vanishing_point(lines)
        if vp is None:
            return None

        m_base, c_base, furthest_points = fit_line_through_furthest_endpoints(lines, vp)
        if m_base is None:
            return None

        # Original lines
        intersections = []
        line_params = []
        for (x1, y1), (x2, y2) in lines:
            dx = x2 - x1
            if abs(dx) < 1e-6:  # Vertical line
                x_inter = x1
                params = (x1, None)
            else:
                m = (y2 - y1) / dx
                if abs(m) < 1e-6:  # Skip horizontal lines
                    continue
                c = y1 - m * x1
                x_inter = (y_point - c) / m
                params = (m, c)
            intersections.append(x_inter)
            line_params.append(params)

        # If no valid lines
        if not line_params:
            return None

        # Separate intersections into left and right
        left = [(x, params) for x, params in zip(intersections, line_params) if x < x_point]
        right = [(x, params) for x, params in zip(intersections, line_params) if x > x_point]

        # Compute perpendicular distance to a line
        def point_to_line_distance(point, line_params):
            x_p, y_p = point
            if line_params[1] is None:  # Vertical line: x = c
                return abs(x_p - line_params[0])
            m, c = line_params
            return abs(m * x_p - y_p + c) / np.sqrt(m**2 + 1)

        # Case 1: Point lies between two lines (considering all)
        if left and right:
            cl_left, left_params = max(left, key=lambda x: x[0])
            cl_right, right_params = min(right, key=lambda x: x[0])
            d_left = point_to_line_distance(point, left_params)
            d_right = point_to_line_distance(point, right_params)
            return d_left + d_right

        # Case 2: Point does not lie between two lines, find closest line by min perp dist
        distances = [point_to_line_distance(point, params) for params in line_params]
        return min(distances)


    if maneuver_type == ManeuverType.ENT:
        # --- End-of-Maneuver Detection (STOP) ---
        front_end_point = parking_coords[0]
        dists_from_front = np.linalg.norm(parking_coords - front_end_point, axis=1)

        rear_end_idx = np.argmax(dists_from_front)
        rear_end_point = parking_coords[rear_end_idx]
        dists_from_rear = np.linalg.norm(parking_coords - rear_end_point, axis=1)
        close_indices = np.where(dists_from_rear <= movement_threshold)[0]
        candidate_frames = parking_df.iloc[close_indices]['frame']
        adjusted_rear_idx = close_indices[np.argmin(candidate_frames)]
        rear_end_point = parking_coords[adjusted_rear_idx]

        interpolated_point = front_end_point*(1-end_entry_ratio) + rear_end_point*end_entry_ratio

        dists_interpolated = np.linalg.norm(parking_coords - interpolated_point, axis=1)
        final_end_idx = np.argmin(dists_interpolated)
        final_end_frame = parking_df.iloc[final_end_idx]['frame']

        # --- Start-of-Maneuver Detection ---
        peak_idx = None
        consecutive_post_peak_frames=0
        for idx in range(tA, len(driving_coords)-tA):
            r1 = driving_coords[idx] - driving_coords[idx-tA]
            r2 = driving_coords[idx+tA] - driving_coords[idx]
            if is_reversal(r1, r2):
                true_reversal = True
                for t in range(1, tA):
                    rA = driving_coords[idx] - driving_coords[max(0, idx-tA-t)]
                    rB = driving_coords[min(len(driving_coords)-1, idx+tA+t)] - driving_coords[idx]
                    if not is_reversal(rA, rB):
                        true_reversal = False
                        break
                if true_reversal:      
                    peak_idx = idx    
            if peak_idx is not None and not is_reversal(r1, r2):
                consecutive_post_peak_frames+=1
            else:
                consecutive_post_peak_frames=0
            if consecutive_post_peak_frames>=tA:
                break

        if(peak_idx is None):
            consecutive_turn_frames = 0
            for idx in range(len(driving_coords)-1, 0, -1):
                aspect_ratio1 = driving_df.iloc[idx-1]['height']/driving_df.iloc[idx-1]['width']
                aspect_ratio2 = driving_df.iloc[idx]['height']/driving_df.iloc[idx]['width']
                if(round(aspect_ratio1, 2)!=round(aspect_ratio2, 2)):
                    consecutive_turn_frames+=1
                if(consecutive_turn_frames==tA):
                    peak_idx = idx

        if on_border(driving_df.iloc[peak_idx]):
            for idx in range(peak_idx, len(driving_df)):
                if not on_border(driving_df.iloc[idx]):
                    peak_idx = idx
                    break
                
        peak_frame = driving_df.iloc[peak_idx]['frame']
        peak_point = driving_coords[peak_idx]

        bottom_peak_point = peak_point.copy()
        bottom_peak_point[1]+=driving_df.iloc[peak_idx]['height']//2

        threshold_distance = find_shortest_distance(bottom_peak_point, parking_lines)
        #print(threshold_distance)
        zone_based_start_idx = 0
        accumulated_distance = 0
        for i in range(peak_idx - 1, -1, -1):
            accumulated_distance+=np.linalg.norm(driving_coords[i+1]-driving_coords[i])
            if(accumulated_distance>=threshold_distance and
               np.linalg.norm(driving_coords[i]-driving_coords[peak_idx])>=threshold_distance):
                zone_based_start_idx = i
                break  

        zone_based_start_frame = driving_df.iloc[zone_based_start_idx]["frame"]

        return (zone_based_start_frame, peak_frame, final_end_frame) 

    if(maneuver_type==ManeuverType.EXT):
        # detect start of exit maneuver 
        front_start_point = driving_coords[0]
        dists_from_front = np.linalg.norm(parking_coords - front_start_point, axis=1)

        rear_start_idx = np.argmax(dists_from_front)
        rear_start_point = parking_coords[rear_start_idx]

        dists_from_rear = np.linalg.norm(parking_coords - rear_start_point, axis=1)
        close_indices = np.where(dists_from_rear <= movement_threshold)[0]
        candidate_frames = parking_df.iloc[close_indices]['frame']
        adjusted_rear_idx = close_indices[np.argmax(candidate_frames)]
        rear_start_point = parking_coords[adjusted_rear_idx]

        interpolated = front_start_point*(start_exit_ratio) + rear_start_point*(1-start_exit_ratio)

        dists_interpolated = np.linalg.norm(parking_coords - interpolated, axis=1)
        final_start_idx = np.argmin(dists_interpolated)
        final_start_frame = parking_df.iloc[final_start_idx]['frame']

        # detect end of exit maneuver
        peak_idx = None
        consecutive_post_peak_frames=0
        for idx in range(tA, len(driving_coords)-tA):
            r1 = driving_coords[idx] - driving_coords[idx-tA]
            r2 = driving_coords[idx+tA] - driving_coords[idx]
            if is_reversal(r1, r2):
                true_reversal = True
                for t in range(1, tA):
                    rA = driving_coords[idx] - driving_coords[max(0, idx-tA-t)]
                    rB = driving_coords[min(len(driving_coords)-1, idx+tA+t)] - driving_coords[idx]
                    if not is_reversal(rA, rB):
                        true_reversal = False
                        break
                if true_reversal:      
                    peak_idx = idx    
            if peak_idx is not None and not is_reversal(r1, r2):
                consecutive_post_peak_frames+=1
            else:
                consecutive_post_peak_frames=0
            if consecutive_post_peak_frames>=tA:
                break

        if(peak_idx is None):
            peak_idx = 0
            consecutive_turn_frames = 0
            for idx in range(len(driving_coords)-1):
                aspect_ratio1 = driving_df.iloc[idx]['height']/driving_df.iloc[idx]['width']
                aspect_ratio2 = driving_df.iloc[idx+1]['height']/driving_df.iloc[idx+1]['width']
                if(round(aspect_ratio1, 2)!=round(aspect_ratio2, 2)):
                    consecutive_turn_frames+=1
                if(consecutive_turn_frames==tA):
                    peak_idx = idx

        if on_border(driving_df.iloc[peak_idx]):
            for idx in range(peak_idx-1, -1, -1):
                if not on_border(driving_df.iloc[idx]):
                    peak_idx = idx
                    break

        peak_frame = driving_df.iloc[peak_idx]['frame']
        peak_point = driving_coords[peak_idx]

        bottom_peak_point = peak_point.copy()
        bottom_peak_point[1]+=driving_df.iloc[peak_idx]['height']//2
    
        threshold_distance = find_shortest_distance(bottom_peak_point, parking_lines)
        zone_based_end_idx = -1
        accumulated_distance = 0
        for i in range(peak_idx, len(driving_coords)-1):
            accumulated_distance+=np.linalg.norm(driving_coords[i+1]-driving_coords[i])
            if(accumulated_distance>=threshold_distance and
                np.linalg.norm(driving_coords[i+1]-driving_coords[peak_idx])>=threshold_distance):
                zone_based_end_idx = i+1
                break

        zone_based_end_frame = driving_df.iloc[zone_based_end_idx]["frame"]

        return (zone_based_end_frame, peak_frame, final_start_frame)


In [2]:
enter_IDs = ['1ENT', '3ENT', '9ENT', '11ENT', '21ENT', '90ENT', '91ENT', '93ENT',
             '94ENT', '95ENT', '96ENT', '97ENT', '98ENT', '290ENT', '291ENT', '292ENT']
exit_IDs = ['0EXT', '2EXT', '4EXT', '80EXT', '81EXT', '83EXT', '84EXT', '85EXT', '86EXT']

In [3]:
enter_start_annotations = []
enter_zbs_predictions = []
enter_peak_predictions = []
for id in enter_IDs:
  prediction_frames = obtain_estimates(id)
  maneuvers = pd.read_csv(f'clip-annotations/maneuver{id}.csv')
  start_frame = np.median(maneuvers["start_frame"])
  enter_start_annotations.append(start_frame)
  enter_zbs_predictions.append(prediction_frames[0])
  enter_peak_predictions.append(prediction_frames[1])
  print(id)
  print((prediction_frames[1]-start_frame)/30.0)

enter_start_annotations = np.array(enter_start_annotations)
enter_zbs_predictions = np.array(enter_zbs_predictions)
enter_peak_predictions = np.array(enter_peak_predictions)

1ENT
0.8333333333333334
3ENT
0.8333333333333334
9ENT
0.6
11ENT
0.6666666666666666
21ENT
0.9
90ENT
0.6
91ENT
-0.5666666666666667
93ENT
0.6666666666666666
94ENT
1.1
95ENT
1.5333333333333334
96ENT
0.6333333333333333
97ENT
0.9666666666666667
98ENT
1.3666666666666667
290ENT
0.1
291ENT
0.4
292ENT
0.5


In [4]:
exit_end_annotations = []
exit_zbe_predictions = []
exit_peak_predictions = []
for id in exit_IDs:
  prediction_frames = obtain_estimates(id)
  maneuvers = pd.read_csv(f'clip-annotations/maneuver{id}.csv')
  end_frame = np.median(maneuvers["end_frame"])
  exit_end_annotations.append(end_frame)
  exit_zbe_predictions.append(prediction_frames[0])
  exit_peak_predictions.append(prediction_frames[1])
  print((prediction_frames[0]-end_frame)/30.0)

exit_end_annotations = np.array(exit_end_annotations)
exit_zbe_predictions = np.array(exit_zbe_predictions)
exit_peak_predictions = np.array(exit_peak_predictions)

-0.7
-0.5666666666666667
0.03333333333333333
0.23333333333333334
0.43333333333333335
0.7
-0.26666666666666666
-0.1
0.3333333333333333


In [5]:
from scipy.optimize import minimize_scalar
def mse_enter(c, y, p, z):
    prediction = c * p + (1 - c) * z
    error = prediction - y
    return np.mean(error ** 2)

def mse_exit(c, y, p, z):
    prediction = c * z + (1 - c) * p
    error = prediction - y
    return np.mean(error ** 2)

res_enter = minimize_scalar(mse_enter, bounds=(0, 1), 
                      args=(enter_start_annotations, enter_zbs_predictions, enter_peak_predictions), 
                      method='bounded')

res_exit = minimize_scalar(mse_exit, bounds=(0, 1),
                           args=(exit_end_annotations, exit_zbe_predictions, exit_peak_predictions),
                           method='bounded')

In [6]:
print(f'res_enter.x: {res_enter.x}')
print(f'res_enter.fun: {np.sqrt(res_enter.fun)/30.0}')
print(f'res_exit.x: {res_exit.x}')
print(f'res_exit.fun: {np.sqrt(res_exit.fun)/30.0}')

res_enter.x: 0.5650503444621093
res_enter.fun: 0.4527831252535774
res_exit.x: 0.017738600500595267
res_exit.fun: 0.4331380075133055
